# Chapter 11 Lab: Frames, Noise, and Quantization

This notebook accompanies **Chapter 11**. It builds the additive-noise
Monte Carlo machinery, the scalar quantization simulator, and the
gradient-descent tight-frame constructor (for m > 5 in higher
dimensions), then works through all five lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import null_space

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_operator(F):
    return F @ F.T

def canonical_dual(F):
    return np.linalg.solve(F @ F.T, F)

def regular_polygon(m):
    angles = 2 * np.pi * np.arange(m) / m
    return np.vstack([np.cos(angles), np.sin(angles)])

def dual_family_direction(F):
    return null_space(F)

def scalar_quantize(c, delta):
    return delta * np.round(c / delta)

def frame_potential(V):
    S = V @ V.T
    return np.trace(S @ S)

def fp_gradient(V):
    S = V @ V.T
    return 4 * S @ V

def normalize_columns(V):
    norms = np.linalg.norm(V, axis=0)
    return V / norms

def tight_unit_norm_frame(n, m, num_steps=1500, lr=0.03, seed=0):
    rng = np.random.default_rng(seed)
    V = rng.standard_normal((n, m))
    V = normalize_columns(V)
    for step in range(num_steps):
        V = V - lr * fp_gradient(V)
        V = normalize_columns(V)
    return V

## Lab Exercise 11.1: Reproducing the Noise Floor Table

For $n=2$, $m=2,3,4,6,8,12,20$: compute predicted noise floor $\sigma^2n^2/m$, Monte Carlo MSE ($\sigma=0.3$, $10^5$ trials), and gain over ONB. Verify gain equals $m/n$.

In [ ]:
sigma = 0.3
n = 2
x_test = np.array([1.0, -0.5])

print(f"{'m':>3} {'predicted':>10} {'MC_mean':>10} {'gain':>8} {'m/n':>6}")
for m in [2, 3, 4, 6, 8, 12, 20]:
    if m == 2:
        F = np.eye(2)
    else:
        F = regular_polygon(m)
    G = None          # TODO: canonical_dual(F)
    predicted = None  # TODO: sigma**2 * np.trace(G @ G.T)

    rng = np.random.default_rng(42)
    errs = []
    for _ in range(100_000):
        eps = rng.normal(0, sigma, size=m)
        x_hat = G @ (F.T @ x_test + eps)
        errs.append(np.sum((x_hat - x_test)**2))
    mc_mean = np.mean(errs)

    onb_error = sigma**2 * n
    gain = onb_error / mc_mean
    print(f"{m:>3} {predicted:>10.5f} {mc_mean:>10.5f} {gain:>8.3f} {m/n:>6.2f}")

## Lab Exercise 11.2: Non-Tight Frame Penalty

$\{h_1,h_2,h_3\}$ with $h_3=(e_1+e_2)/\sqrt2$: (a) compute $\|\tilde H\|_F^2$. (b) compare to $n^2/m=4/3$. (c) Monte Carlo verification. (d) explain geometrically why non-tight is worse.

In [ ]:
h1 = np.array([1.0, 0.0])
h2 = np.array([0.0, 1.0])
h3 = np.array([1.0, 1.0]) / np.sqrt(2)
F_h = np.column_stack([h1, h2, h3])

H_tilde = None  # TODO: canonical_dual(F_h)
frob_sq = None  # TODO: np.trace(H_tilde @ H_tilde.T)
print(f"||H_tilde||_F^2 = {frob_sq:.6f}")

tight_prediction = 4/3
print(f"Tight unit-norm prediction n^2/m = {tight_prediction:.6f}")
print(f"Non-tight is {'WORSE (larger)' if frob_sq > tight_prediction else 'better (smaller)'}")

sigma = 0.3
x_test = np.array([1.0, -0.5])
rng = np.random.default_rng(7)
errs = []
for _ in range(100_000):
    eps = rng.normal(0, sigma, size=3)
    x_hat = H_tilde @ (F_h.T @ x_test + eps)
    errs.append(np.sum((x_hat - x_test)**2))
mc_mean = np.mean(errs)
predicted = sigma**2 * frob_sq
print(f"\nPredicted MSE: {predicted:.6f}, Monte Carlo mean: {mc_mean:.6f}")

S_h = F_h @ F_h.T
eigs_h = np.linalg.eigvalsh(S_h)
print(f"\nFrame operator eigenvalues: {eigs_h} (A={eigs_h.min():.4f}, B={eigs_h.max():.4f})")

*Your explanation for part (d): what is it about the frame's geometry (not just the formula) that causes the larger dual Frobenius norm?* (replace this text)

## Lab Exercise 11.3: Quantization Simulation

For $n=2$, $m\in\{3,4,6,8,12\}$: (a) verify the worst-case bound $n^2\delta^2/4$ never violated. (b) plot empirical max vs. bound vs. $n^2\delta^2/(4m)$. (c) explain, and construct a constructively-aligned example.

In [ ]:
delta = 0.2
n = 2
proved_bound = (n**2 * delta**2) / 4

ms = [3, 4, 6, 8, 12]
empirical_maxes = []
typical_errors = []

for m in ms:
    F = regular_polygon(m)
    G = canonical_dual(F)

    rng = np.random.default_rng(1)
    X = rng.standard_normal((n, 100_000))
    X = X / np.linalg.norm(X, axis=0)

    C_true = F.T @ X
    C_q = None                # TODO: scalar_quantize(C_true, delta)
    errs = np.sum((G @ C_q - X)**2, axis=0)

    emp_max = errs.max()
    empirical_maxes.append(emp_max)
    typical_errors.append(n**2 * delta**2 / (4*m))

    print(f"m={m}: empirical_max={emp_max:.6f}, proved_bound={proved_bound:.6f}, "
          f"holds={emp_max <= proved_bound + 1e-9}")

plt.figure(figsize=(7,5))
plt.plot(ms, empirical_maxes, 'o-', label='empirical max error')
plt.axhline(proved_bound, color='red', linestyle='--', label='proved bound (constant)')
plt.plot(ms, typical_errors, 's--', label='typical error n^2 delta^2/(4m)')
plt.xlabel('m')
plt.ylabel('error')
plt.legend()
plt.title('Quantization error: empirical max vs. proved bound vs. typical')
plt.show()

In [ ]:
F3 = regular_polygon(3)
G3 = canonical_dual(F3)

delta_test = 0.3
rng2 = np.random.default_rng(3)
best_ratio = 0
best_x = None
for _ in range(2000):
    x_try = rng2.standard_normal(2)
    x_try = x_try / np.linalg.norm(x_try)
    c_true = F3.T @ x_try
    c_q = scalar_quantize(c_true, delta_test)
    actual_err = np.sum((G3 @ c_q - x_try)**2)
    typical = 2**2 * delta_test**2 / (4*3)
    ratio = actual_err / typical
    if ratio > best_ratio:
        best_ratio = ratio
        best_x = x_try

print(f"Best x found: {best_x}")
print(f"Ratio of actual to typical error: {best_ratio:.4f}")

*Your explanation: why is the proved bound constant in m while the empirical maximum decays?* (replace this text)

## Lab Exercise 11.4: Higher Dimensions

Extend to $n=3$, $m=6,9,12,18$ using `tight\_unit\_norm\_frame`. Verify $\|\tilde F\|_F^2\approx n^2/m$, run Monte Carlo, plot noise floor vs. m.

In [ ]:
n = 3
sigma = 0.3
ms_3d = [6, 9, 12, 18]
noise_floors = []

for m in ms_3d:
    F = None  # TODO: tight_unit_norm_frame(n, m, num_steps=1500, lr=0.03, seed=0)
    G = canonical_dual(F)
    frob_sq = np.trace(G @ G.T)
    predicted_ratio = n**2 / m

    print(f"m={m}: ||F_tilde||_F^2={frob_sq:.5f}, n^2/m={predicted_ratio:.5f}, "
          f"match={np.isclose(frob_sq, predicted_ratio, atol=1e-2)}")

    noise_floor = sigma**2 * frob_sq
    noise_floors.append(noise_floor)

plt.figure(figsize=(7,5))
plt.loglog(ms_3d, noise_floors, 'o-', label='noise floor (n=3)')
ref = noise_floors[0] * ms_3d[0] / np.array(ms_3d)
plt.loglog(ms_3d, ref, '--', color='gray', label='theoretical ~1/m')
plt.xlabel('m'); plt.ylabel('noise floor')
plt.legend()
plt.title('Noise floor decay for fixed n=3')
plt.show()

## Lab Exercise 11.5 (Challenge): Alternate Duals and Noise

(a) For triangular frame, $G(w)=\tilde F+wk^T$, $\|G(w)\|_F^2=4/3+\|w\|^2$. (b) At $\|w\|=1$, compare noise floor to canonical. (c) At $w^*=(1,1)/\sqrt3$ for $\{h_1,h_2,h_3\}$, compute $\|G(w^*)\|_F^2$.

In [ ]:
f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])
F_tri_tilde = canonical_dual(F_tri)
k_tri = dual_family_direction(F_tri)[:, 0]

sigma = 0.3

w_unit = np.array([1.0, 0.0])
G_w = None       # TODO: F_tri_tilde + np.outer(w_unit, k_tri)
energy_w = None  # TODO: np.trace(G_w @ G_w.T)
print(f"Energy at ||w||=1: {energy_w:.6f}  (expect 4/3 + 1 = {4/3+1:.6f})")

canonical_floor = sigma**2 * (4/3)
floor_at_w = sigma**2 * energy_w
pct_penalty = (floor_at_w - canonical_floor) / canonical_floor * 100
print(f"Canonical noise floor: {canonical_floor:.6f}")
print(f"Noise floor at ||w||=1: {floor_at_w:.6f}")
print(f"Percentage penalty: {pct_penalty:.2f}%")

h1, h2, h3 = np.array([1.0,0.0]), np.array([0.0,1.0]), np.array([1.0,1.0])
F_h_orig = np.column_stack([h1, h2, h3])
F_h_tilde = canonical_dual(F_h_orig)
k_h = dual_family_direction(F_h_orig)[:, 0]

w_star = np.array([1.0, 1.0]) / np.sqrt(3)
G_wstar = None       # TODO: F_h_tilde + np.outer(w_star, k_h)
energy_wstar = None  # TODO: np.trace(G_wstar @ G_wstar.T)

canonical_floor_h = sigma**2 * np.trace(F_h_tilde @ F_h_tilde.T)
floor_at_wstar = sigma**2 * energy_wstar
print(f"\n{{h1,h2,h3}} canonical noise floor: {canonical_floor_h:.6f}")
print(f"{{h1,h2,h3}} noise floor at w*: {floor_at_wstar:.6f}")

*Your reflection: how does this illustrate that minimizing total noise and minimizing noise anisotropy generally require different dual choices?* (replace this text)